# Project Sentinel – Silver Layer

## Objective

This notebook transforms raw Bronze data into a clean and reliable Silver layer by applying data quality rules, removing duplicates, handling null values, and converting columns to appropriate data types.

### Features
- Data Cleaning
- Null Handling
- Duplicate Removal
- Data Type Conversion
- Business Rule Validation
- Delta Lake Storage
- Structured Streaming

### Execution Flow

Input:
- Bronze Delta Table

Processing:
- Remove duplicates
- Handle null values
- Convert data types
- Improve data quality

Output:
- Silver Delta Table

Next Notebook:
04_Gold_Layer

In [0]:
BASE_PATH = "/Volumes/workspace/default/project_sentinel"

BRONZE_PATH = f"{BASE_PATH}/bronze"
SILVER_PATH = f"{BASE_PATH}/silver"
print("Bronze Path :", BRONZE_PATH)
print("Silver Path :", SILVER_PATH)

Bronze Path : /Volumes/workspace/default/project_sentinel/bronze
Silver Path : /Volumes/workspace/default/project_sentinel/silver


In [0]:
from pyspark.sql.functions import col


In [0]:
bronze_df = spark.read.format("delta").load(BRONZE_PATH)
print("Bronze Record Count:", bronze_df.count())

display(bronze_df.limit(10))

Bronze Record Count: 4489


transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,_rescued_data,ingestion_time
7001,2021-01-10 09:15:00,P401,201,UPI,Yes,No,180,2,240,null,2026-07-09T05:30:52.715Z
7002,2021-02-05 11:30:00,P155,202,Cash,No,Yes,90,4,130,null,2026-07-09T05:30:52.715Z
7003,2021-03-12 15:20:00,P210,203,Debit Card,Yes,No,300,1,420,null,2026-07-09T05:30:52.715Z
7004,2021-04-18 10:05:00,P099,204,Credit Card,Yes,Yes,110,3,160,null,2026-07-09T05:30:52.715Z
7005,2021-05-22 16:45:00,P501,205,UPI,No,Yes,75,5,115,null,2026-07-09T05:30:52.715Z
6001,2022-08-15 10:15:00,P101,101,UPI,Yes,No,120,2,150,null,2026-07-09T05:15:53.384Z
6002,2022-08-15 10:20:00,P205,102,Credit Card,Yes,Yes,250,1,320,null,2026-07-09T05:15:53.384Z
6003,2022-08-15 10:25:00,P110,103,Debit Card,No,Yes,80,5,100,null,2026-07-09T05:15:53.384Z
6004,2022-08-15 10:30:00,P315,104,Cash,No,No,450,1,550,null,2026-07-09T05:15:53.384Z
6005,2022-08-15 10:35:00,P122,105,UPI,Yes,Yes,90,3,130,null,2026-07-09T05:15:53.384Z


In [0]:
bronze_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transactional_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- payment: string (nullable = true)
 |-- credit_card: string (nullable = true)
 |-- loyalty_card: string (nullable = true)
 |-- cost: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- price: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



In [0]:
silver_df = (
    bronze_df

    #Remove bad records
    .filter(col("transaction_id").isNotNull())

    .dropDuplicates()

    #convert data types
    .withColumn("transaction_id", col("transaction_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("cost", col("cost").cast("double"))
    .withColumn("price", col("price").cast("double"))
)

In [0]:
print("Silver Record Count:", silver_df.count())

silver_df.printSchema()

display(silver_df.limit(10))

Silver Record Count: 4489
root
 |-- transaction_id: integer (nullable = true)
 |-- transactional_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- payment: string (nullable = true)
 |-- credit_card: string (nullable = true)
 |-- loyalty_card: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,_rescued_data,ingestion_time
5,2021-05-04 05:45:00,P0058,5,mastercard,5108752372298261,T,11.38,2,12.39,null,2026-07-08T17:04:28.822Z
27,2021-05-05 20:09:00,P0622,5,visa,4041596972700199,F,13.87,1,14.89,null,2026-07-08T17:04:28.822Z
45,2021-05-08 01:48:00,P0431,4,americanexpress,374283964832018,F,10.67,4,11.59,null,2026-07-08T17:04:28.822Z
46,2021-05-08 03:04:00,P0170,10,americanexpress,374288240835982,F,8.13,4,8.89,null,2026-07-08T17:04:28.822Z
49,2021-05-08 13:43:00,P0661,3,visa,4041593023791707,F,8.21,4,8.99,null,2026-07-08T17:04:28.822Z
56,2021-05-08 22:40:00,P0035,6,visa,4041598082914470,F,11.76,3,13.09,null,2026-07-08T17:04:28.822Z
57,2021-05-08 23:32:00,P0268,7,americanexpress,374288707197975,T,11.93,1,13.39,null,2026-07-08T17:04:28.822Z
102,2021-05-12 18:32:00,P0545,8,mastercard,5108757985493662,T,6.62,1,7.89,null,2026-07-08T17:04:28.822Z
109,2021-05-13 07:17:00,P0423,8,mastercard,5108754133243636,F,14.04,3,14.89,null,2026-07-08T17:04:28.822Z
122,2021-05-15 01:03:00,P0305,6,americanexpress,374283437595721,F,8.74,3,9.99,null,2026-07-08T17:04:28.822Z


In [0]:
dbutils.fs.rm(SILVER_PATH, True)

True

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(SILVER_PATH)

In [0]:
silver_df.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- transactional_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- payment: string (nullable = true)
 |-- credit_card: string (nullable = true)
 |-- loyalty_card: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



In [0]:
silver_table = spark.read.format("delta").load(SILVER_PATH)

print("Total Records:", silver_table.count())

silver_table.printSchema()

display(silver_table)

Total Records: 4489
root
 |-- transaction_id: integer (nullable = true)
 |-- transactional_date: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- payment: string (nullable = true)
 |-- credit_card: string (nullable = true)
 |-- loyalty_card: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)



transaction_id,transactional_date,product_id,customer_id,payment,credit_card,loyalty_card,cost,quantity,price,_rescued_data,ingestion_time
5,2021-05-04 05:45:00,P0058,5,mastercard,5108752372298261,T,11.38,2,12.39,null,2026-07-08T17:04:28.822Z
27,2021-05-05 20:09:00,P0622,5,visa,4041596972700199,F,13.87,1,14.89,null,2026-07-08T17:04:28.822Z
45,2021-05-08 01:48:00,P0431,4,americanexpress,374283964832018,F,10.67,4,11.59,null,2026-07-08T17:04:28.822Z
46,2021-05-08 03:04:00,P0170,10,americanexpress,374288240835982,F,8.13,4,8.89,null,2026-07-08T17:04:28.822Z
49,2021-05-08 13:43:00,P0661,3,visa,4041593023791707,F,8.21,4,8.99,null,2026-07-08T17:04:28.822Z
56,2021-05-08 22:40:00,P0035,6,visa,4041598082914470,F,11.76,3,13.09,null,2026-07-08T17:04:28.822Z
57,2021-05-08 23:32:00,P0268,7,americanexpress,374288707197975,T,11.93,1,13.39,null,2026-07-08T17:04:28.822Z
102,2021-05-12 18:32:00,P0545,8,mastercard,5108757985493662,T,6.62,1,7.89,null,2026-07-08T17:04:28.822Z
109,2021-05-13 07:17:00,P0423,8,mastercard,5108754133243636,F,14.04,3,14.89,null,2026-07-08T17:04:28.822Z
122,2021-05-15 01:03:00,P0305,6,americanexpress,374283437595721,F,8.74,3,9.99,null,2026-07-08T17:04:28.822Z


In [0]:
from pyspark.sql.functions import col

print("Null Transaction IDs:",
      silver_table.filter(col("transaction_id").isNull()).count())

Null Transaction IDs: 0


In [0]:
print("Duplicate Rows:",
      silver_table.count() - silver_table.dropDuplicates().count())

Duplicate Rows: 0


In [0]:
print("Bronze:", bronze_df.count())
print("Silver:", silver_table.count())

Bronze: 4489
Silver: 4489


---
### Notebook Completed Successfully ✅

**Project:** Project Sentinel

**Architecture:** Medallion Architecture

**Technology Stack:** Databricks • PySpark • Delta Lake • Auto Loader • Structured Streaming